# 03 — Feature Engineering

Purpose: create robust behavioral, IAM, and resource-scope features.
Raw identifiers (`user_id`, `role_id`), raw resource ARNs, and exact AWS action names are retained for audit/inference but excluded from the predictive feature set later.

In [1]:
# AI-Based IAM Permission Optimizer — Model V2
# Run notebooks in order: 01 → 10
# Raw CSVs should be available in the project root or adjust RAW_DIR below.
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

df = pd.read_csv(DATA_DIR / "cleaned_dataset.csv")

for col in ["first_used", "last_used"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

print("Loaded cleaned data:", df.shape)

Loaded cleaned data: (20000, 27)


In [2]:
ACTION_VERBS = [
    "describe", "get", "list", "head", "read", "put", "create", "update",
    "modify", "delete", "remove", "attach", "detach", "invoke", "start",
    "stop", "terminate", "send", "receive", "publish", "subscribe", "pass",
    "assume", "tag", "untag"
]

def extract_action_family(action):
    name = str(action).split(":", 1)[-1].lower()
    for verb in ACTION_VERBS:
        if name.startswith(verb):
            return verb
    return "other"

# Duration of observed activity
df["active_days"] = (
    (df["last_used"] - df["first_used"]).dt.total_seconds() / 86400.0
).clip(lower=0)

# Frequency and stabilized failure rate
df["usage_frequency"] = (
    df["usage_count"] / df["unique_days_used"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan)

df["smoothed_failure_rate"] = (
    df["failure_count"] + 1
) / (df["usage_count"] + 2)

# Log transforms reduce sensitivity to extreme usage counts.
df["log_usage_count"] = np.log1p(df["usage_count"].clip(lower=0))
df["log_unique_days_used"] = np.log1p(df["unique_days_used"].clip(lower=0))

# Recompute success rate from counts rather than trusting duplicated input columns.
df["success_rate"] = np.where(
    df["usage_count"] > 0,
    df["success_count"] / df["usage_count"],
    0.0
).clip(0, 1)

# Action metadata
df["action_family"] = df["action"].map(extract_action_family).astype("string")
df["is_delete"] = df["operation_type"].astype("string").str.upper().eq("DELETE").astype(int)
df["is_permission_change"] = df["operation_type"].astype("string").str.upper().eq("PERMISSION_CHANGE").astype(int)
df["is_security_sensitive"] = df["operation_type"].astype("string").str.upper().eq("SECURITY_SENSITIVE").astype(int)

In [3]:
# Resource-scope features
scope = df.get("resource_scope", pd.Series("unknown", index=df.index)).astype("string").str.strip().str.lower()
df["is_wildcard_resource"] = scope.eq("*").astype(int)
df["scope_breadth"] = np.select(
    [scope.eq("*"), scope.eq("specific")],
    [1.0, 0.0],
    default=0.5
)

# Do not fill missing behavioral values here: preprocessing will fit imputers only on training data.
feature_preview_cols = [
    c for c in [
        "usage_count", "log_usage_count", "unique_days_used",
        "log_unique_days_used", "days_since_last_use", "active_days",
        "usage_frequency", "success_rate", "smoothed_failure_rate",
        "service", "operation_type", "risk_level", "risk_weight",
        "action_family", "is_delete", "is_permission_change",
        "is_security_sensitive", "is_wildcard_resource", "scope_breadth"
    ] if c in df.columns
]

display(df[feature_preview_cols].head())

,usage_count,log_usage_count,unique_days_used,log_unique_days_used,days_since_last_use,active_days,usage_frequency,success_rate,smoothed_failure_rate,service,operation_type,risk_level,risk_weight,action_family,is_delete,is_permission_change,is_security_sensitive,is_wildcard_resource,scope_breadth
0,4,1.609438,4,1.609438,111,8.0,1.000000,1.000000,0.166667,API Gateway,TAGGING,LOW,3,tag,0,0,0,0,0.0
1,19,2.995732,10,2.397895,30,31.0,1.900000,1.000000,0.047619,API Gateway,READ,LOW,2,get,0,0,0,0,0.0
2,43,3.784190,8,2.197225,34,28.0,5.375000,1.000000,0.022222,RDS,WRITE,MEDIUM,5,create,0,0,0,0,0.0
3,21,3.091042,17,2.890372,24,25.0,1.235294,1.000000,0.043478,S3,PERMISSION_CHANGE,CRITICAL,10,put,0,1,0,1,1.0
4,38,3.663562,18,2.944439,15,57.0,2.111111,0.973684,0.050000,API Gateway,WRITE,MEDIUM,5,update,0,0,0,0,0.0


In [4]:
engineered_path = DATA_DIR / "engineered_dataset.csv"
df.to_csv(engineered_path, index=False)

print("Saved:", engineered_path)
print("Engineered shape:", df.shape)

Saved: C:\Users\LENOVO\Downloads\IAM_Model_V2_Notebooks\data\engineered_dataset.csv
Engineered shape: (20000, 38)
